In [4]:
# %% [markdown]
# # Step 1: Test Your Existing Model
# ## Verify that your .h5 model loads correctly with custom metrics

# %%
import tensorflow as tf
import numpy as np
import pickle
from tensorflow.keras import layers, models, backend as K
import os
import shutil
import sys

# %% [markdown]
# ### Define custom metrics (RMSE, R2, MAE) that were used in training

# %%
# Define custom metrics functions
def rmse(y_true, y_pred):
    """Root Mean Square Error"""
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def r2(y_true, y_pred):
    """R-squared (Coefficient of determination)"""
    SS_res = K.sum(K.square(y_true - y_pred))
    SS_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - SS_res/(SS_tot + K.epsilon())

def mae(y_true, y_pred):
    """Mean Absolute Error"""
    return K.mean(K.abs(y_pred - y_true))

# Also define the metric names that might be in the model
def root_mean_squared_error(y_true, y_pred):
    return rmse(y_true, y_pred)

def r_squared(y_true, y_pred):
    return r2(y_true, y_pred)

def mean_absolute_error(y_true, y_pred):
    return mae(y_true, y_pred)

# ============= ADD CUSTOM ACTIVATION FUNCTIONS =============
def mish_activation(x):
    """
    Mish activation function: x * tanh(softplus(x))
    """
    return x * tf.math.tanh(tf.math.softplus(x))

def swish_activation(x):
    """
    Swish activation function: x * sigmoid(x)
    """
    return x * tf.nn.sigmoid(x)

# Register the custom activations with Keras
tf.keras.utils.get_custom_objects()['mish_activation'] = mish_activation
tf.keras.utils.get_custom_objects()['swish_activation'] = swish_activation
# ============================================================

# %% [markdown]
# ### Helper function to get model path

# %%
def get_model_path():
    """
    Get the model path from the models folder in the project
    """
    # Check if model exists in models folder
    model_path = os.path.join('models', 'Pyramid_fusion_densenet121_model.h5')
    
    if os.path.exists(model_path):
        print(f"✅ Model found at: {model_path}")
        return model_path
    else:
        print(f"❌ Model not found at: {model_path}")
        print("   Please ensure Pyramid_fusion_densenet121_model.h5 is in the 'models' folder")
        return None

# %% [markdown]
# ### Load model with custom objects

# %%
try:
    # Get model path from project folder
    model_path = get_model_path()
    
    if model_path is None:
        raise FileNotFoundError("No model file found in 'models' folder")
    
    print(f"📦 Loading model from: {model_path}")
    
    # Define comprehensive custom objects dictionary with all metrics and custom activations
    custom_objects = {
        # Loss functions - ALL using class-based approach
        'mse': tf.keras.losses.MeanSquaredError(),
        'MSE': tf.keras.losses.MeanSquaredError(),
        'mean_squared_error': tf.keras.losses.MeanSquaredError(),
        'mae': mae,
        'MAE': mae,
        'mean_absolute_error': mae,
        'rmse': rmse,
        'RMSE': rmse,
        'root_mean_squared_error': rmse,
        'r2': r2,
        'R2': r2,
        'r_squared': r2,
        'R_squared': r2,
        # Custom activation functions
        'mish_activation': mish_activation,
        'swish_activation': swish_activation,
        # The function objects themselves (Keras might look for these)
        mish_activation: mish_activation,
        swish_activation: swish_activation,
    }
    
    # Load model with custom objects - use compile=False first to avoid metric issues
    model = tf.keras.models.load_model(
        model_path, 
        custom_objects=custom_objects,
        compile=False
    )
    
    print("✅ Model loaded successfully with custom objects!")
    
    # Recompile the model with custom metrics
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),  # Use class-based here too
        metrics=[mae, rmse, r2]
    )
    print("✅ Model recompiled with custom metrics!")
    print(f"Model type: {type(model)}")
    
except Exception as e:
    print(f"❌ First attempt failed: {e}")
    import traceback
    traceback.print_exc()
    
    # Second attempt: Load with compile=False and minimal custom objects including activations
    try:
        # Get model path again
        model_path = get_model_path()
        if model_path is None:
            raise FileNotFoundError("No model file found")
        
        # Minimal custom objects including activations
        minimal_custom_objects = {
            'mse': tf.keras.losses.MeanSquaredError(),
            'mean_squared_error': tf.keras.losses.MeanSquaredError(),
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'mish_activation': mish_activation,
            'swish_activation': swish_activation,
        }
        
        model = tf.keras.models.load_model(
            model_path, 
            custom_objects=minimal_custom_objects,
            compile=False
        )
        print("✅ Model loaded with minimal custom objects!")
        
        # Try to infer optimizer and compile
        model.compile(
            optimizer='adam',
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[mae, rmse, r2]
        )
        print("✅ Model recompiled successfully!")
        
    except Exception as e2:
        print(f"❌ Second attempt failed: {e2}")
        traceback.print_exc()
        
        # Third attempt: Last resort - load without compilation but with custom activations
        try:
            # Get model path again
            model_path = get_model_path()
            if model_path is None:
                raise FileNotFoundError("No model file found")
            
            # Register custom activations globally
            tf.keras.utils.get_custom_objects()['mish_activation'] = mish_activation
            tf.keras.utils.get_custom_objects()['swish_activation'] = swish_activation
            
            model = tf.keras.models.load_model(model_path, compile=False)
            print("✅ Model loaded with compile=False!")
            
            # Add custom objects as attributes
            model.rmse = rmse
            model.r2 = r2
            model.mae = mae
            model.mish_activation = mish_activation
            model.swish_activation = swish_activation
            
            print("⚠️ Model loaded but custom objects are attached as attributes")
            print("   You may need to use them manually during evaluation")
            
        except Exception as e3:
            print(f"❌ Third attempt failed: {e3}")
            traceback.print_exc()
            model = None

# %%
# If model loaded successfully, inspect its architecture
if model is not None:
    print("\n" + "="*60)
    print("✅ MODEL LOADED SUCCESSFULLY!")
    print("="*60)
    
    # Print model summary
    print("\n📋 Model Summary:")
    model.summary()
    
    # Check input details
    print("\n" + "="*60)
    print("📊 Model Input Details:")
    print("="*60)
    
    # Handle multiple inputs (for multimodal models)
    if hasattr(model, 'input'):
        if isinstance(model.input, list):
            for i, input_layer in enumerate(model.input):
                print(f"\nInput {i+1}:")
                print(f"  Name: {input_layer.name}")
                print(f"  Shape: {input_layer.shape}")
                print(f"  Dtype: {input_layer.dtype}")
        else:
            print(f"\nSingle Input:")
            print(f"  Name: {model.input.name}")
            print(f"  Shape: {model.input.shape}")
            print(f"  Dtype: {model.input.dtype}")
    else:
        print("\n⚠️ Could not access model.input directly")
        print("   Model may have been loaded in a different format")
    
    # Check output details
    print("\n" + "="*60)
    print("📊 Model Output Details:")
    print("="*60)
    
    if hasattr(model, 'output'):
        if isinstance(model.output, list):
            for i, output_layer in enumerate(model.output):
                print(f"\nOutput {i+1}:")
                print(f"  Name: {output_layer.name}")
                print(f"  Shape: {output_layer.shape}")
        else:
            print(f"\nSingle Output:")
            print(f"  Name: {model.output.name}")
            print(f"  Shape: {model.output.shape}")
    
    # Check model metrics
    print("\n" + "="*60)
    print("📊 Model Metrics:")
    print("="*60)
    if hasattr(model, 'metrics_names'):
        for i, metric_name in enumerate(model.metrics_names):
            print(f"Metric {i+1}: {metric_name}")
    
    # %% [markdown]
    # ### Test with appropriate dummy data

    # %%
    print("\n" + "="*60)
    print("🧪 Testing Model with Dummy Data")
    print("="*60)
    
    try:
        # Prepare dummy inputs based on actual model input structure
        if hasattr(model, 'input') and isinstance(model.input, list):
            # Multimodal model with multiple inputs
            dummy_inputs = []
            for i, input_layer in enumerate(model.input):
                # Get the expected shape (excluding batch dimension)
                input_shape = input_layer.shape[1:]
                print(f"\nPreparing dummy data for Input {i+1} with shape: {input_shape}")
                
                # Create dummy data with correct shape
                dummy_data = np.random.rand(1, *input_shape).astype(np.float32)
                dummy_inputs.append(dummy_data)
            
            # Make prediction
            predictions = model.predict(dummy_inputs, verbose=0)
            
        elif hasattr(model, 'input'):
            # Single input model
            input_shape = model.input.shape[1:]
            print(f"\nPreparing dummy data for Single Input with shape: {input_shape}")
            dummy_input = np.random.rand(1, *input_shape).astype(np.float32)
            predictions = model.predict(dummy_input, verbose=0)
            
        else:
            # Try to infer input shape from model layers
            print("\n⚠️ Could not access model.input, trying alternative method...")
            # Get first layer's input shape
            if model.layers:
                first_layer = model.layers[0]
                if hasattr(first_layer, 'input_shape'):
                    input_shape = first_layer.input_shape[1:]
                    print(f"  Inferred input shape: {input_shape}")
                    dummy_input = np.random.rand(1, *input_shape).astype(np.float32)
                    predictions = model.predict(dummy_input, verbose=0)
                else:
                    raise Exception("Could not infer input shape")
            else:
                raise Exception("Model has no layers")
        
        print(f"\n✅ Test prediction successful!")
        
        if isinstance(predictions, list):
            for i, pred in enumerate(predictions):
                print(f"Prediction {i+1} shape: {pred.shape}")
                print(f"Sample values: {pred[0][:5] if len(pred[0]) > 5 else pred[0]}")
        else:
            print(f"Predictions shape: {predictions.shape}")
            print(f"Sample prediction values: {predictions[0][:5]}")
        
    except Exception as e:
        print(f"\n❌ Test prediction failed: {e}")
        print("\n💡 Tip: The error might be due to incorrect input shape.")
        print("Check the model input shapes above and adjust the dummy data accordingly.")

    # %% [markdown]
    # ### Save model configuration with custom metrics info

    # %%
    # Save model configuration to a text file
    config_path = 'models/model_config.txt'
    
    try:
        with open(config_path, 'w', encoding='utf-8') as f:
            f.write("MODEL CONFIGURATION\n")
            f.write("="*60 + "\n\n")
            
            # Write custom metrics info
            f.write("CUSTOM METRICS USED:\n")
            f.write("-"*30 + "\n")
            f.write("RMSE (Root Mean Square Error)\n")
            f.write("R² (R-squared)\n")
            f.write("MAE (Mean Absolute Error)\n\n")
            
            # Write custom activation functions
            f.write("CUSTOM ACTIVATIONS USED:\n")
            f.write("-"*30 + "\n")
            f.write("Mish Activation (mish_activation)\n")
            f.write("Swish Activation (swish_activation)\n\n")
            
            # Write input information
            f.write("INPUTS:\n")
            f.write("-"*30 + "\n")
            if hasattr(model, 'input') and isinstance(model.input, list):
                for i, input_layer in enumerate(model.input):
                    f.write(f"Input {i+1}: {input_layer.name}\n")
                    f.write(f"  Shape: {input_layer.shape}\n")
                    f.write(f"  Dtype: {input_layer.dtype}\n\n")
            elif hasattr(model, 'input'):
                f.write(f"Single Input: {model.input.name}\n")
                f.write(f"  Shape: {model.input.shape}\n")
                f.write(f"  Dtype: {model.input.dtype}\n\n")
            else:
                f.write("Input information not available\n\n")
            
            # Write output information
            f.write("OUTPUTS:\n")
            f.write("-"*30 + "\n")
            if hasattr(model, 'output') and isinstance(model.output, list):
                for i, output_layer in enumerate(model.output):
                    f.write(f"Output {i+1}: {output_layer.name}\n")
                    f.write(f"  Shape: {output_layer.shape}\n\n")
            elif hasattr(model, 'output'):
                f.write(f"Single Output: {model.output.name}\n")
                f.write(f"  Shape: {model.output.shape}\n\n")
            else:
                f.write("Output information not available\n\n")
            
            # Write model metrics
            if hasattr(model, 'metrics_names'):
                f.write("METRICS:\n")
                f.write("-"*30 + "\n")
                for metric_name in model.metrics_names:
                    f.write(f"• {metric_name}\n")
                f.write("\n")
        
        print(f"\n✅ Model configuration saved to: {config_path}")
        
    except Exception as e:
        print(f"\n❌ Error saving configuration: {e}")

else:
    print("\n❌ Could not load model. Please check the model file and errors above.")
    

# %% [markdown]
# # Additional Analysis for Web App: Understanding Model Inputs
# ## Determine exact input shapes and feature requirements

# %%
# Load your trained model (if not already loaded)
if model is None:
    try:
        # Register custom activations before loading
        tf.keras.utils.get_custom_objects()['mish_activation'] = mish_activation
        tf.keras.utils.get_custom_objects()['swish_activation'] = swish_activation
        
        # Get model path from project folder
        model_path = get_model_path()
        if model_path:
            model = tf.keras.models.load_model(model_path, compile=False)
            print("✅ Model loaded for analysis")
        else:
            print("❌ No model found in models folder")
    except Exception as e:
        print(f"❌ Could not load model for analysis: {e}")
        model = None

# %%
if model is not None:
    print("="*60)
    print("MODEL INPUT ANALYSIS FOR WEB APPLICATION")
    print("="*60)

    # Analyze model inputs
    print("\n📊 MODEL INPUTS:")
    print("-"*40)
    
    if hasattr(model, 'input') and isinstance(model.input, list):
        for i, input_layer in enumerate(model.input):
            print(f"\nInput {i+1}:")
            print(f"  Name: {input_layer.name}")
            print(f"  Shape: {input_layer.shape}")
            print(f"  Expected features: {input_layer.shape[1] if len(input_layer.shape) > 1 else 'N/A'}")
    elif hasattr(model, 'input'):
        print(f"\nSingle Input:")
        print(f"  Name: {model.input.name}")
        print(f"  Shape: {model.input.shape}")
        print(f"  Expected features: {model.input.shape[1] if len(model.input.shape) > 1 else 'N/A'}")
    else:
        # Try to get input shape from first layer
        if model.layers:
            first_layer = model.layers[0]
            if hasattr(first_layer, 'input_shape'):
                input_shape = first_layer.input_shape
                print(f"\nInferred Input Shape (from first layer): {input_shape}")
                if isinstance(input_shape, list):
                    for i, shape in enumerate(input_shape):
                        print(f"  Input {i+1}: {shape}")
                else:
                    print(f"  Single Input: {input_shape}")

# %%
if model is not None:
    # Determine which features are used
    print("\n" + "="*60)
    print("FEATURE MAPPING FOR WEB APP")
    print("="*60)

    print("\nDuring training, these 4 tabular features were used:")
    print("  1. Air Temperature (Avg_Temp) - Will be provided by farmers")
    print("  2. Soil Temperature (Soil) - NOT available to farmers")
    print("  3. Nitrogen Fertilizer Level - AVAILABLE to farmers")
    print("  4. Number of Days - AVAILABLE to farmers")

    print("\n✅ For web application, we will:")
    print("  • Use Nitrogen Fertilizer Level (from dropdown)")
    print("  • Use Number of Days (calculated from dates)")
    print("  • Use Air Temperature (user input)")
    print("  • Set Soil Temperature = mean value from training data")

# %%
# Calculate mean value from your training data for soil temperature only
# Air temperature will be provided by farmers
print("\n" + "="*60)
print("SOIL TEMPERATURE VALUE FOR WEB APP")
print("="*60)

# Replace this with actual mean from your training data
avg_soil_temp_mean = 29.8  # Example value - REPLACE WITH YOUR ACTUAL MEAN

print(f"\nBased on training data:")
print(f"  • Mean Soil Temperature: {avg_soil_temp_mean}°C")

print("\n📝 For web app predictions, we will:")
print(f"  • Set Soil Temperature = {avg_soil_temp_mean}°C (mean value from training)")
print("  • Use actual Air Temperature from user input")
print("  • Use actual Nitrogen Level from user input")
print("  • Use actual Days from date calculation")

# %%
# Save only soil temperature mean for use in Flask app
temperature_values = {
    'soil_temp_mean': avg_soil_temp_mean
}

# Save to file for Flask app to use
import pickle
import os

# Create utils directory if it doesn't exist
os.makedirs('utils', exist_ok=True)

with open('utils/temperature_means.pkl', 'wb') as f:
    pickle.dump(temperature_values, f)
print("\n✅ Soil temperature mean saved to utils/temperature_means.pkl")

# %%
# Verify the saved file
print("\n" + "="*60)
print("VERIFYING SAVED SOIL TEMPERATURE")
print("="*60)

try:
    with open('utils/temperature_means.pkl', 'rb') as f:
        loaded_values = pickle.load(f)
    print(f"✅ Successfully loaded: {loaded_values}")
    print(f"   Soil Temperature Mean: {loaded_values['soil_temp_mean']}°C")
except Exception as e:
    print(f"❌ Error loading: {e}")

✅ Model found at: models\Pyramid_fusion_densenet121_model.h5
📦 Loading model from: models\Pyramid_fusion_densenet121_model.h5
✅ Model loaded successfully with custom objects!
✅ Model recompiled with custom metrics!
Model type: <class 'keras.src.models.functional.Functional'>

✅ MODEL LOADED SUCCESSFULLY!

📋 Model Summary:


Model: "Enhanced_Multimodal_FPN_Attention_DenseNet121_transfer_learning"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ zero_padding2d_6              │ (None, 230, 230, 3)       │               0 │ image_input[0][0]          │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,408 │ zero_padding2d_6[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ zero_padding2d_7              │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1 (MaxPooling2D)          │ (None, 56, 56, 64)        │               0 │ zero_padding2d_7[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_bn             │ (None, 56, 56, 64)        │             256 │ pool1[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_0_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 128)       │           8,192 │ conv2_block1_0_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 128)       │             512 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 128)       │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 32)        │          36,864 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_concat           │ (None, 56, 56, 96)        │               0 │ pool1[0][0],               │
│ (Concatenate)                 │                           │               

 Total params: 9,786,625 (37.33 MB)

 Trainable params: 2,742,081 (10.46 MB)

 Non-trainable params: 7,044,544 (26.87 MB)


📊 Model Input Details:

Input 1:
  Name: image_input
  Shape: (None, 224, 224, 3)
  Dtype: float32

Input 2:
  Name: tabular_input
  Shape: (None, 4)
  Dtype: float32

📊 Model Output Details:

Single Output:
  Name: keras_tensor_3050
  Shape: (None, 1)

📊 Model Metrics:
Metric 1: loss
Metric 2: compile_metrics

🧪 Testing Model with Dummy Data

Preparing dummy data for Input 1 with shape: (224, 224, 3)

Preparing dummy data for Input 2 with shape: (4,)

✅ Test prediction successful!
Predictions shape: (1, 1)
Sample prediction values: [-1.3741902]

✅ Model configuration saved to: models/model_config.txt
MODEL INPUT ANALYSIS FOR WEB APPLICATION

📊 MODEL INPUTS:
----------------------------------------

Input 1:
  Name: image_input
  Shape: (None, 224, 224, 3)
  Expected features: 224

Input 2:
  Name: tabular_input
  Shape: (None, 4)
  Expected features: 4

FEATURE MAPPING FOR WEB APP

During training, these 4 tabular features were used:
  1. Air Temperature (Avg_Temp) - Will be provided